In [ ]:
import tifffile
from PIL import Image
from matplotlib import pyplot as plt
import numpy as np
import cv2

# Download Data

download data from https://www.10xgenomics.com/products/xenium-in-situ/preview-dataset-human-breast

# Load Morphology Images

## Print ImageDescription

In [ ]:
mip_url = "PATH/TO/morphology_mip.ome.tif"
with tifffile.TiffFile(mip_url) as tif:
    for tag in tif.pages[0].tags.values():
        if tag.name == 'ImageDescription':
            print(tag.name+":", tag.value)

## Print Dimensions of the Pyramid Level images  

In [ ]:
with tifffile.TiffFile(mip_url) as tif:
    for i in range(len(tif.series[0].levels)):
        print(i, tif.series[0].levels[i].shape)

## Plot Level 3 DAPI Image

In [ ]:
with tifffile.TiffFile(mip_url) as tif:
    DAPI3_image = tif.series[0].levels[3].asarray()
    

plt.imshow(DAPI3_image, cmap='binary')
plt.title('Level 3 DAPI', size=16)
plt.axis('scaled')
plt.show()

DAPI3im = Image.fromarray(DAPI3_image)
# DAPI3im.save("Xe_hsa_breast_c_Level_3_DAPI_morphology_mip-3.tif")

# Load H&E Images

## Print ImageDescription

In [ ]:
he_url = "PATH/TO/Xenium_FFPE_Human_Breast_Cancer_Rep1_he_image.ome.tif"
with tifffile.TiffFile(he_url) as tif:
    for tag in tif.pages[0].tags.values():
        if tag.name == 'ImageDescription':
            print(tag.name+":", tag.value)

## Print Dimensions of the Pyramid Level Images  

In [ ]:
with tifffile.TiffFile(he_url) as tif:
    for i in range(len(tif.series[0].levels)):
        print(i, tif.series[0].levels[i].shape)

## Plot Level 3 H&E Image

In [ ]:
with tifffile.TiffFile(he_url) as tif:
    HE3_image = tif.series[0].levels[3].asarray()
    

plt.imshow(HE3_image, cmap='binary')
plt.title('Level 3 H&E', size=16)
plt.axis('scaled')
plt.show()

HE3im = Image.fromarray(HE3_image)
# HE3im.save("Xe_hsa_breast_c_Level_3_HE-3.tif")

# Align HE to DAPI/Transcript Coordinate System

## load Transformation Matrix Exported from Xenium Explorer or Downloaded from Dataset

In [ ]:
img_alignment_url = "PATH/TO/Xenium_FFPE_Human_Breast_Cancer_Rep1_he_imagealignment.csv"
img_alignment_m = np.genfromtxt(img_alignment_url,delimiter = ',')
img_alignment_m

## Load Level 0 H&E Image

In [ ]:
he_url = "PATH/TO/Xenium_FFPE_Human_Breast_Cancer_Rep1_he_image.ome.tif"
with tifffile.TiffFile(he_url) as tif:
    HE0_image = tif.series[0].levels[0].asarray()

In [ ]:
with tifffile.TiffFile(mip_url) as tif:
    dapi0_shape = tif.series[0].levels[0].shape
    dapi3_shape = tif.series[0].levels[3].shape

## Coordonate System Transfrom

In [ ]:
HE0_image_ali = cv2.warpPerspective(HE0_image, img_alignment_m, dapi0_shape[1::-1], borderMode=cv2.BORDER_REFLECT_101) 
plt.imshow(HE0_image_ali)

## Resize Image

In [ ]:
imgResize = cv2.resize(HE0_image_ali, dapi3_shape[1::-1])
plt.imshow(imgResize)

In [ ]:
tifffile.imwrite('./he-raw.tiff', imgResize)